In [2]:

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBClassifier
import joblib
import os
from pathlib import Path

# Find the project root directory
ROOT = Path.cwd()

if not (ROOT / "datasets").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "datasets"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("All libraries imported!")
print("Project root:", ROOT)
print("Dataset folder:", DATA_DIR)
print("Model folder:", MODEL_DIR)


All libraries imported!
Project root: E:\Agro-Agent-Enhanced
Dataset folder: E:\Agro-Agent-Enhanced\datasets
Model folder: E:\Agro-Agent-Enhanced\models


In [3]:
df = pd.read_csv(DATA_DIR / "Crop_recommendation.csv")

print("Crop dataset loaded!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(3))

Crop dataset loaded!
Shape: (2200, 8)
Columns: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']
    N   P   K  temperature   humidity        ph    rainfall label
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice


In [4]:
X = df[['N', 'P', 'K', 'temperature',
        'humidity', 'ph', 'rainfall']]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

crop_model = RandomForestClassifier(
    n_estimators=100, random_state=42
)
crop_model.fit(X_train, y_train)

y_pred = crop_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("Crop Model Accuracy:", round(acc * 100, 2), "%")
print(classification_report(y_test, y_pred))

joblib.dump(crop_model, MODEL_DIR / "crop_model.pkl")
print("Crop model saved!")

Crop Model Accuracy: 99.55 %
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      0.95      0.97        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.95      1.00      0.98        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       1.00      1.00      1.00        20
       maize       0.95      1.00      0.98        20
       mango       1.00      1.00      1.00        20
   mothbeans       1.00      1.00      1.00        20
    mungbean       1.00      1.00      1.00        20
   muskmelon       1.00      1.00      1.00        20
      orange       1.00      1.00      1.00        2

In [27]:
# Load the original fertilizer dataset
fert_check = pd.read_csv(DATA_DIR / "Fertilizer Prediction.csv")

# Remove whitespace from column names
fert_check.columns = fert_check.columns.str.strip()

print("Original dataset shape:", fert_check.shape)
print("\nOriginal fertilizer counts:")
print(fert_check["Fertilizer Name"].value_counts())

# Exclude the suspicious date-like label
fert_clean = fert_check[
    fert_check["Fertilizer Name"].astype(str).str.strip()
    != "10/26/2026"
].copy()

# Remove duplicate rows
fert_clean = fert_clean.drop_duplicates().copy()

print("\nCleaned dataset shape:", fert_clean.shape)
print("\nCleaned fertilizer counts:")
print(fert_clean["Fertilizer Name"].value_counts())

print("\nRemaining duplicates:", fert_clean.duplicated().sum())

Original dataset shape: (198, 9)

Original fertilizer counts:
Fertilizer Name
Urea          44
DAP           36
28-28         34
14-35-14      28
20-20         28
17-17-17      14
10/26/2026    14
Name: count, dtype: int64

Cleaned dataset shape: (92, 9)

Cleaned fertilizer counts:
Fertilizer Name
Urea        22
DAP         18
28-28       17
14-35-14    14
20-20       14
17-17-17     7
Name: count, dtype: int64

Remaining duplicates: 0


In [28]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

train_df = fert_clean.copy()

# Create encoders
le_soil = LabelEncoder()
le_crop = LabelEncoder()
le_fert = LabelEncoder()

# Encode input categories
train_df["Soil Type"] = le_soil.fit_transform(
    train_df["Soil Type"].astype(str)
)

train_df["Crop Type"] = le_crop.fit_transform(
    train_df["Crop Type"].astype(str)
)

# Encode the target
train_df["Fertilizer Name"] = le_fert.fit_transform(
    train_df["Fertilizer Name"].astype(str)
)

# Keep the feature order consistent with the backend
fert_features = [
    "Temparature",
    "Humidity",
    "Moisture",
    "Soil Type",
    "Crop Type",
    "Nitrogen",
    "Potassium",
    "Phosphorous"
]

X2 = train_df[fert_features]
y2 = train_df["Fertilizer Name"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2,
    y2,
    test_size=0.2,
    random_state=42,
    stratify=y2
)

print("Training rows:", len(X2_train))
print("Testing rows:", len(X2_test))
print("Number of fertilizer classes:", len(le_fert.classes_))
print("Fertilizer classes:", list(le_fert.classes_))

Training rows: 73
Testing rows: 19
Number of fertilizer classes: 6
Fertilizer classes: ['14-35-14', '17-17-17', '20-20', '28-28', 'DAP', 'Urea']


In [29]:
fert_model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric="mlogloss"
)

fert_model.fit(X2_train, y2_train)

print("Fertilizer model training completed!")

Fertilizer model training completed!


In [30]:
from sklearn.metrics import accuracy_score, classification_report

y2_train_pred = fert_model.predict(X2_train)
y2_test_pred = fert_model.predict(X2_test)

print("Training accuracy:",
      round(accuracy_score(y2_train, y2_train_pred) * 100, 2), "%")

print("Testing accuracy:",
      round(accuracy_score(y2_test, y2_test_pred) * 100, 2), "%")

print("\nClassification report:")
print(
    classification_report(
        y2_test,
        y2_test_pred,
        labels=range(len(le_fert.classes_)),
        target_names=le_fert.classes_,
        zero_division=0
    )
)

Training accuracy: 100.0 %
Testing accuracy: 100.0 %

Classification report:
              precision    recall  f1-score   support

    14-35-14       1.00      1.00      1.00         3
    17-17-17       1.00      1.00      1.00         1
       20-20       1.00      1.00      1.00         3
       28-28       1.00      1.00      1.00         3
         DAP       1.00      1.00      1.00         4
        Urea       1.00      1.00      1.00         5

    accuracy                           1.00        19
   macro avg       1.00      1.00      1.00        19
weighted avg       1.00      1.00      1.00        19



In [31]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from xgboost import XGBClassifier

X_raw = fert_clean.drop(columns=["Fertilizer Name"]).copy()
y_raw = fert_clean["Fertilizer Name"].astype(str)

target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y_raw)

categorical_cols = ["Soil Type", "Crop Type"]
numeric_cols = [c for c in X_raw.columns if c not in categorical_cols]

preprocessor = ColumnTransformer([
    ("categories", OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=-1
    ), categorical_cols),
    ("numbers", "passthrough", numeric_cols)
])

cv_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric="mlogloss"
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    cv_model, X_raw, y_encoded, cv=cv, scoring="accuracy"
)

print("Fold accuracies:", (scores * 100).round(2))
print("Mean CV accuracy:", round(scores.mean() * 100, 2), "%")
print("Standard deviation:", round(scores.std() * 100, 2), "%")

Fold accuracies: [ 94.74 100.   100.    88.89 100.  ]
Mean CV accuracy: 96.73 %
Standard deviation: 4.42 %


In [32]:
import joblib

MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(fert_model, MODEL_DIR / "fert_model.pkl")
joblib.dump(le_soil, MODEL_DIR / "le_soil.pkl")
joblib.dump(le_crop, MODEL_DIR / "le_crop.pkl")
joblib.dump(le_fert, MODEL_DIR / "le_fert.pkl")

print("Fertilizer model and encoders saved successfully!")
print("Saved to:", MODEL_DIR)

Fertilizer model and encoders saved successfully!
Saved to: E:\Agro-Agent-Enhanced\models


In [34]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

# 1. Prepare the dataset
df_yield = yield2.drop(
    columns=["Unnamed: 0"],
    errors="ignore"
).copy()

# 2. Remove missing values
df_yield = df_yield.dropna().copy()

# 3. Encode categorical columns
le_area = LabelEncoder()
le_item = LabelEncoder()

df_yield["Area"] = le_area.fit_transform(
    df_yield["Area"].astype(str)
)

df_yield["Item"] = le_item.fit_transform(
    df_yield["Item"].astype(str)
)

# 4. Select features and target
X3 = df_yield[
    [
        "Area",
        "Item",
        "Year",
        "average_rain_fall_mm_per_year",
        "pesticides_tonnes",
        "avg_temp"
    ]
]

y3 = df_yield["hg/ha_yield"]

# 5. Split the data
X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3,
    y3,
    test_size=0.2,
    random_state=42
)

# 6. Train the model
yield_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

yield_model.fit(X3_train, y3_train)

print("Yield model training completed!")

# 7. Evaluate the model
y3_pred = yield_model.predict(X3_test)

r2 = r2_score(y3_test, y3_pred)
rmse = mean_squared_error(y3_test, y3_pred) ** 0.5

print("Yield Model R2 Score:", round(r2, 4))
print("Yield Model RMSE:", round(rmse, 2))

# 8. Save the model and encoders
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(yield_model, MODEL_DIR / "yield_model.pkl")
joblib.dump(le_area, MODEL_DIR / "le_area.pkl")
joblib.dump(le_item, MODEL_DIR / "le_item.pkl")

print("Yield model and encoders saved successfully!")
print("Saved to:", MODEL_DIR)


Yield model training completed!
Yield Model R2 Score: 0.9857
Yield Model RMSE: 10181.76
Yield model and encoders saved successfully!
Saved to: E:\Agro-Agent-Enhanced\models


In [35]:
for file in sorted(MODEL_DIR.glob("*.pkl")):
    print(file.name)

crop_model.pkl
fert_model.pkl
le_area.pkl
le_crop.pkl
le_fert.pkl
le_item.pkl
le_soil.pkl
yield_model.pkl


In [12]:
!pip install matplotlib

  Using cached matplotlib-3.10.8-cp310-cp310-win_amd64.whl (8.1 MB)
  Using cached kiwisolver-1.5.0-cp310-cp310-win_amd64.whl (73 kB)
  Using cached contourpy-1.3.2-cp310-cp310-win_amd64.whl (221 kB)
  Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
     ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
     - -------------------------------------- 0.0/1.6 MB 281.8 kB/s eta 0:00:06
     - -------------------------------------- 0.1/1.6 MB 328.2 kB/s eta 0:00:05
     -- ------------------------------------- 0.1/1.6 MB 547.6 kB/s eta 0:00:03
     -- ------------------------------------- 0.1/1.6 MB 547.6 kB/s eta 0:00:03
     -- ------------------------------------- 0.1/1.6 MB 547.6 kB/s eta 0:00:03
     --- ------------------------------------ 0.2/1.6 MB 416.


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
import joblib

print("=== Saved model files ===")
for file in sorted(MODEL_DIR.glob("*.pkl")):
    print(file.name)

print("\n=== Fertilizer encoder classes ===")
print("Soil types:", le_soil.classes_)
print("Crop types:", le_crop.classes_)
print("Fertilizer types:", le_fert.classes_)

print("\n=== Yield encoder classes ===")
print("Areas:", le_area.classes_)
print("Yield crop types:", le_item.classes_)

=== Saved model files ===
crop_model.pkl
fert_model.pkl
le_area.pkl
le_crop.pkl
le_fert.pkl
le_item.pkl
le_soil.pkl
yield_model.pkl

=== Fertilizer encoder classes ===
Soil types: ['Black' 'Clayey' 'Loamy' 'Red' 'Sandy']
Crop types: ['Barley' 'Cotton' 'Ground Nuts' 'Maize' 'Millets' 'Oil seeds' 'Paddy'
 'Pulses' 'Sugarcane' 'Tobacco' 'Wheat']
Fertilizer types: ['14-35-14' '17-17-17' '20-20' '28-28' 'DAP' 'Urea']

=== Yield encoder classes ===
Areas: ['Albania' 'Algeria' 'Angola' 'Argentina' 'Armenia' 'Australia' 'Austria'
 'Azerbaijan' 'Bahamas' 'Bahrain' 'Bangladesh' 'Belarus' 'Belgium'
 'Botswana' 'Brazil' 'Bulgaria' 'Burkina Faso' 'Burundi' 'Cameroon'
 'Canada' 'Central African Republic' 'Chile' 'Colombia' 'Croatia'
 'Denmark' 'Dominican Republic' 'Ecuador' 'Egypt' 'El Salvador' 'Eritrea'
 'Estonia' 'Finland' 'France' 'Germany' 'Ghana' 'Greece' 'Guatemala'
 'Guinea' 'Guyana' 'Haiti' 'Honduras' 'Hungary' 'India' 'Indonesia' 'Iraq'
 'Ireland' 'Italy' 'Jamaica' 'Japan' 'Kazakhstan' 'Ke